# 5 · Discover, download, and add a layer to GeoLibre

This standalone tutorial discovers one Sentinel-2 Item through Earth Search, selects one conservative public HTTPS data-role COG candidate, and renders explicit actions for the provider file and [GeoLibre Web](https://web.geolibre.app/). Run all cells in JupyterLite. Metadata requests begin when you run the connection/search cell; raster bytes are **not** requested until you choose an action. If discovery fails because of CORS, TLS, HTTP status, timeout, or provider outage, the client surfaces that error. Retry or choose another catalog; the notebook never fabricates results.

In [ ]:
from stac_browser import STACBrowser, PRESETS, explorer_url

client = await STACBrowser(PRESETS['earthsearch']).connect()
items = await client.search(
    bbox=[-123.1, 37.2, -121.8, 38.1],
    collection='sentinel-2-l2a',
    start='2024-06-01', end='2024-06-15', limit=1)
if not items:
    raise RuntimeError('No Items matched. Retry later or broaden the bounded dates/area.')
item = items[0]
print('Discovered Item:', item['id'])
print('Collection:', item.get('collection'))
print('Available assets:', list(item.get('assets', {})))

## Select one exact public COG candidate

The helper uses the same conservative public-asset boundary as notebook 04. It does not claim that a TIFF suffix proves cloud optimization. GeoLibre still validates the actual bytes, and protected or signed assets remain provider workflows.

In [ ]:
handoff = client.public_cog_handoff(item, explorer=explorer_url())
selected_asset = item['assets'][handoff['asset_key']]
collection_metadata = next((entry for entry in client.collections if entry.get('id') == item.get('collection')), {})
print('Selected candidate asset:', handoff['asset_key'])
print('Asset title:', selected_asset.get('title'))
print('Asset media type:', selected_asset.get('type'))
print('Asset roles:', selected_asset.get('roles'))
print('Collection license:', collection_metadata.get('license', 'not advertised'))
print('Collection providers:', collection_metadata.get('providers', []))
print('Exact provider address:', handoff['href'])
print('No raster request has been made by this helper.')

## User-activated compatibility bridge

**Check & add exact asset to GeoLibre** opens this site's compatibility bridge. Only after your click, it requests byte 0 without credentials. A readable HTTP 206 response continues to GeoLibre's exact `data` link. If CORS, TLS, network policy, or Range behavior prevents verification, the bridge offers the original provider Download, local drag/drop or **Add data** instructions, and **Try remote URL anyway**. The check is origin-specific and cannot guarantee the final layer load.

In [ ]:
import json
from pathlib import Path
from html import escape
from IPython.display import HTML, display

display(HTML(
    '<p><a href="' + escape(handoff['download'], quote=True) + '" target="_blank" rel="noopener noreferrer" download>'
    'Download selected GeoTIFF/COG candidate</a></p>'
    '<p><a href="' + escape(handoff['geolibre'], quote=True) + '" target="_blank" rel="noopener noreferrer">'
    'Check &amp; add exact asset to GeoLibre ↗</a></p>'))
client.save('geolibre-tutorial')
for saved_name in ('geolibre-tutorial-items.geojson', 'geolibre-tutorial-search.json'):
    json.loads(Path(saved_name).read_text(encoding='utf-8'))
print('Saved geolibre-tutorial-items.geojson and geolibre-tutorial-search.json')
print('GeoLibre workflow ready')

## Local import and attribution

When remote loading is unavailable, download the selected file, open GeoLibre, then drag the file onto the map or use **Add data** and choose the matching raster layer. Inspect the Item and collection license and retain provider attribution. The full Explorer supports additional GeoLibre formats and provides the same compatibility fallback from its metadata pane.